# 🎮 SteamSense

# 06 - Content Based Recommendation System

## Objective

Build an AI recommendation engine using:

- Game Titles
- Descriptions
- Tags

Algorithms

- TF-IDF
- Cosine Similarity

In [36]:
import pandas as pd

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
DATA_PATH = Path("../data/processed")

games = pd.read_csv(
    DATA_PATH / "games_content.csv"
)

In [38]:
games.head()

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,...,steam_deck,release_year,game_age,is_free,platform_count,popularity_score,rating_score,description,tags,content
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,...,1,2008,18,0,1,358.248506,8,Enter the dark underworld of Prince of Persia ...,Action Adventure Parkour Third Person Great So...,Prince of Persia: Warrior Within™ Enter the da...
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,...,1,2011,15,0,1,174.641698,6,NaN,Action,BRINK: Agents of Change Action
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,...,1,2013,13,0,3,384.091403,8,Monaco: What's Yours Is Mine is a single playe...,Co-op Stealth Indie Heist Local Co-Op Strategy...,Monaco: What's Yours Is Mine Monaco: What's Yo...
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,...,1,2014,12,0,1,307.523215,5,Escape Dead Island is a Survival-Mystery adven...,Zombies Adventure Survival Action Third Person...,Escape Dead Island Escape Dead Island is a Sur...
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,...,1,2014,12,0,2,416.032040,8,Dungeon of the Endless is a Rogue-Like Dungeon...,Roguelike Strategy Tower Defense Pixel Graphic...,Dungeon of the ENDLESS™ Dungeon of the Endless...


In [39]:
games["content"] = (
    games["content"]
    .fillna("")
)

In [40]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

In [41]:
tfidf_matrix = tfidf.fit_transform(
    games["content"]
)

In [42]:
tfidf_matrix.shape

(50872, 10000)

In [43]:
from sklearn.metrics.pairwise import linear_kernel

In [44]:
indices = pd.Series(
    games.index,
    index=games["title"]
).drop_duplicates()

In [45]:
def recommend_games(title, top_n=10):

    if title not in indices:
        return "Game not found."

    idx = indices[title]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    similar_games = cosine_scores.argsort()[-top_n-1:-1][::-1]

    return games.iloc[
        similar_games
    ][
        [
            "title",
            "rating",
            "price_final",
            "positive_ratio"
        ]
    ]

In [46]:
recommend_games(
    "Cyberpunk 2077"
)

,title,rating,price_final,positive_ratio
30663,Cyberpunk 2077 REDmod,Mixed,0.00,66
35917,Cyberpunk Fighting,Positive,0.99,95
45703,Cyberpunk 2077 Bonus Content,Very Positive,0.00,82
13387,NINJASLAYER : AREA 4643,Very Positive,10.99,98
19307,Brigador - Audiobook,Positive,4.99,100
43876,Cyberdrome,Very Positive,0.00,88
44577,House Flipper - Cyberpunk DLC,Very Positive,0.00,83
29818,Cyberpunk Madness,Positive,0.99,80
30718,Chronicles of cyberpunk,Positive,0.00,86
17604,Brigador - Vol. I,Positive,4.99,100


In [47]:
recommend_games(
    "Portal 2"
)

,title,rating,price_final,positive_ratio
15040,Portal 2,Overwhelmingly Positive,10.00,98
13589,Portal Knights - Portal Pioneer Pack,Mixed,2.99,50
45174,Portal Stories: VR,Very Positive,0.00,85
1585,Portal Dungeon,Very Positive,12.99,86
5938,Portal Knights - Weddings and Galas,Positive,0.00,94
24766,Kawaii Rainbow Portal,Mostly Positive,1.99,73
7582,Portal 2 - The Final Hours,Mostly Positive,1.99,78
38847,Portal Stories: Mel,Overwhelmingly Positive,0.00,96
2832,Bridge Constructor Portal - Portal Proficiency,Very Positive,3.99,87
12019,Portal Knights - Lobot Box,Mostly Positive,2.99,70


In [48]:
recommend_games(
    "Terraria"
)

,title,rating,price_final,positive_ratio
11443,Terraria: Otherworld Official Soundtrack,Very Positive,4.99,96
49905,tModLoader,Overwhelmingly Positive,0.00,97
11194,Terraria: Official Soundtrack,Very Positive,4.99,95
33772,Squarelands,Mixed,3.99,54
43963,WorldQuest,Mixed,0.00,68
25993,Dr. Umgebung's School of Life,Mixed,2.99,50
11475,Craft The World,Very Positive,18.99,88
17458,Gamer To Game Developer Series 2: Learn Unity 2D,Mostly Positive,24.99,76
50868,PAYDAY 3,Mostly Negative,40.00,38
50867,I Expect You To Die 3: Cog in the Machine,Very Positive,22.00,96


In [49]:
import joblib
from pathlib import Path

OUTPUT_DIR = Path("../recommendation")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(tfidf, OUTPUT_DIR / "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, OUTPUT_DIR / "tfidf_matrix.pkl")
joblib.dump(games["app_id"], OUTPUT_DIR / "app_ids.pkl")
joblib.dump(games["title"], OUTPUT_DIR / "titles.pkl")
joblib.dump(games[["app_id", "title", "content"]], OUTPUT_DIR / "games.pkl")

print("✅ Recommendation assets saved successfully!")

✅ Recommendation assets saved successfully!
